# Phase 5: Spatiotemporal Graph Neural Network Modeling
### PyTorch Geometric `TrafficGCN` & Residual Change (`TrafficGCN-Delta`) Formulation

This notebook implements spatial graph deep learning:
1. Environment verification and adjacency tensor preparation.
2. Leak-free z-score normalization (standardized strictly on training partition).
3. 3D spatiotemporal graph tensor sequence construction $(N_{\text{samples}}, 207, 12)$.
4. `TrafficGCN` architecture definition (2-layer `GCNConv` with ReLU).
5. Standard GCN training with PyG mini-batching.
6. Momentum-aware residual formulation: predicting velocity changes $\Delta y_t = y_t - x_{t-1}$.
7. De-normalized test set evaluation (MAE and RMSE).


## Section 1: Environment Setup, Dependencies & System Configuration
We configure the runtime environment, set plotting styles, verify GPU hardware acceleration, and import fundamental scientific computing libraries (`numpy`, `pandas`, `matplotlib`).


In [4]:
#Day01_Traffic_Exploration.ipynb
# Basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('ggplot')

print("Setup complete ")


Setup complete 


In [ ]:
# Ingest METR-LA dataset (checks local directory or Colab environment)
import os
import zipfile
import pandas as pd

file_path = 'METR-LA.h5'
zip_path = 'METR-LA.h5.zip'

if not os.path.exists(file_path):
    if os.path.exists(zip_path):
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('.')
    elif os.path.exists('/content/METR-LA.h5.zip'):
        !unzip -o /content/METR-LA.h5.zip -d /content/
        file_path = '/content/METR-LA.h5'

if os.path.exists(file_path):
    df = pd.read_hdf(file_path)
elif os.path.exists('/content/METR-LA.h5'):
    df = pd.read_hdf('/content/METR-LA.h5')
else:
    # Fallback to local or downloaded copy
    df = pd.read_hdf('METR-LA.h5')

print("Dataset Shape (timesteps, sensors):", df.shape)
df.head()


In [ ]:
import urllib.request

base = "https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/"
files = ["distances_la_2012.csv", "graph_sensor_ids.txt", "graph_sensor_locations.csv"]

for fname in files:
    urllib.request.urlretrieve(base + fname, fname)
    print(f"downloaded {fname}")


downloaded distances_la_2012.csv
downloaded graph_sensor_ids.txt
downloaded graph_sensor_locations.csv


In [ ]:
import numpy as np
import pandas as pd

with open('graph_sensor_ids.txt') as f:
    sensor_ids = f.read().strip().split(',')
num_sensors = len(sensor_ids)
id_to_idx = {sid: i for i, sid in enumerate(sensor_ids)}

dist_df = pd.read_csv('distances_la_2012.csv', dtype={'from': str, 'to': str})
print(dist_df.columns.tolist())  # sanity check: should be ['from', 'to', 'cost']

dist_mx = np.full((num_sensors, num_sensors), np.inf)

froms = dist_df['from'].values
tos = dist_df['to'].values
costs = dist_df['cost'].values

for f, t, c in zip(froms, tos, costs):
    if f in id_to_idx and t in id_to_idx:
        dist_mx[id_to_idx[f], id_to_idx[t]] = c

distances = dist_mx[~np.isinf(dist_mx)].flatten()
std = distances.std()
adj_mx = np.exp(-np.square(dist_mx / std))
adj_mx[adj_mx < 0.1] = 0
np.fill_diagonal(adj_mx, 1.0)

print(adj_mx.shape)
print((adj_mx > 0).sum(), "edges (incl. self-loops)")


['from', 'to', 'cost']
(207, 207)
1722 edges (incl. self-loops)


## Section 9: Graph Sequence Preparation & Leak-Free Train-Set Standardization
Before training Graph Neural Networks, data must be formatted into node feature tensors:
- Node feature matrix $X \in \mathbb{R}^{B \times N \times T_{\text{in}}}$, where $B$ is batch size, $N = 207$ nodes, and $T_{\text{in}} = 12$ historical timesteps.
- Target vector $y \in \mathbb{R}^{B \times N}$ corresponding to future traffic speeds.
- **Data Hygiene**: Zero-values (sensor dropouts) are imputed via forward/backward fill. Normalization parameters (mean $\mu$ and standard deviation $\sigma$) are strictly computed on the training split only to prevent data leakage into validation and test sets.

### 9.1 PyTorch & PyG Verification, Graph Tensors Construction


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

!pip install torch_geometric --no-deps -q

import torch

edge_index = np.array(np.nonzero(adj_mx))
edge_weight = adj_mx[edge_index[0], edge_index[1]]

edge_index = torch.tensor(edge_index, dtype=torch.long)
edge_weight = torch.tensor(edge_weight, dtype=torch.float)

print("edge_index shape:", edge_index.shape)   # (2, 1722)
print("edge_weight shape:", edge_weight.shape) # (1722,)


edge_index shape: torch.Size([2, 1722])
edge_weight shape: torch.Size([1722])


### 9.2 Leak-Free Standardization & 3D Graph Sequence Generation


In [ ]:
# Define graph sequence extraction: (N_samples, num_sensors, seq_length)
SEQ_LENGTH = 12

def create_graph_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length].T)   # (num_sensors, seq_length)
        y.append(data[i+seq_length])       # (num_sensors,)
    return np.array(X), np.array(y)

# Impute zeros and compute statistics exclusively on the training partition
df_clean = df.replace(0, np.nan).ffill().bfill()
values_clean = df_clean.values

train_len = int(len(values_clean) * 0.7)
mean = values_clean[:train_len].mean()
std = values_clean[:train_len].std()
values_norm = (values_clean - mean) / std

# Generate spatiotemporal graph sequences
X_graph, y_graph = create_graph_sequences(values_norm, SEQ_LENGTH)

# Chronological partition: 70% Train, 15% Validation, 15% Test
n = len(X_graph)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train_g, y_train_g = X_graph[:train_end], y_graph[:train_end]
X_val_g, y_val_g     = X_graph[train_end:val_end], y_graph[train_end:val_end]
X_test_g, y_test_g   = X_graph[val_end:], y_graph[val_end:]

print(f"Train normalization stats -> Mean: {mean:.4f}, Std: {std:.4f}")
print("Graph Tensor Shapes:")
print("  X_train_g:", X_train_g.shape, "| y_train_g:", y_train_g.shape)
print("  X_val_g:  ", X_val_g.shape,   "| y_val_g:  ", y_val_g.shape)
print("  X_test_g: ", X_test_g.shape,  "| y_test_g: ", y_test_g.shape)


mean: 58.435151943038306 std: 13.098095807372092
X_graph: (34260, 207, 12) y_graph: (34260, 207)
train/val/test: (23982, 207, 12) (5139, 207, 12) (5139, 207, 12)


## Section 10: Graph Convolutional Network (`TrafficGCN`) Architecture & Direct Speed Modeling
Graph Convolutional Networks (Kipf & Welling, 2017) perform first-order localized spectral convolutions on graphs:
$$H^{(l+1)} = \sigma\left(\tilde{D}^{-\frac{1}{2}} \tilde{A} \tilde{D}^{-\frac{1}{2}} H^{(l)} W^{(l)}\right)$$
where $\tilde{A} = A + I_N$ is the adjacency matrix with added self-loops, and $\tilde{D}_{ii} = \sum_j \tilde{A}_{ij}$.

### 10.1 `TrafficGCN` Neural Network Definition (PyTorch Geometric)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
!pip install torch_geometric -q
from torch_geometric.nn import GCNConv

class TrafficGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, out_channels=1):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index, edge_weight):
        # x: (num_nodes, in_channels)
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_weight)
        return x.squeeze(-1)  # (num_nodes,)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.4 MB/s eta 0:00:00


### 10.2 PyG Batching Utility & Standard GCN Training Procedure


In [ ]:
from torch_geometric.data import Data, Batch

def make_batch(X_np, y_np, indices, edge_index, edge_weight):
    """Build a PyG Batch from a set of timestep indices."""
    data_list = []
    for i in indices:
        x = torch.tensor(X_np[i], dtype=torch.float)   # (207, 12)
        y = torch.tensor(y_np[i], dtype=torch.float)    # (207,)
        data_list.append(Data(x=x, edge_index=edge_index, edge_attr=edge_weight, y=y))
    return Batch.from_data_list(data_list)

def train_gcn(X_train_g, y_train_g, X_val_g, y_val_g, edge_index, edge_weight,
              epochs=20, batch_size=64, lr=0.01, hidden=64, device=None):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TrafficGCN(in_channels=X_train_g.shape[2], hidden_channels=hidden, out_channels=1).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    edge_index_d = edge_index.to(device)
    edge_weight_d = edge_weight.to(device)

    n_train = X_train_g.shape[0]

    for epoch in range(epochs):
        model.train()
        perm = np.random.permutation(n_train)
        total_loss = 0.0

        for i in range(0, n_train, batch_size):
            idx = perm[i:i+batch_size]
            batch = make_batch(X_train_g, y_train_g, idx, edge_index, edge_weight).to(device)

            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.edge_attr)  # (batch_size*207,)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)

        avg_train_loss = total_loss / n_train

        # validation
        model.eval()
        with torch.no_grad():
            val_idx = np.random.choice(X_val_g.shape[0], size=min(500, X_val_g.shape[0]), replace=False)
            val_batch = make_batch(X_val_g, y_val_g, val_idx, edge_index, edge_weight).to(device)
            val_out = model(val_batch.x, val_batch.edge_index, val_batch.edge_attr)
            val_loss = criterion(val_out, val_batch.y).item()

        print(f"Epoch {epoch+1}/{epochs} - train MAE: {avg_train_loss:.4f} - val MAE (sample): {val_loss:.4f}")

    return model


### 10.3 Training Direct GCN Model (20 Epochs) & Test Evaluation


In [ ]:
model = train_gcn(X_train_g, y_train_g, X_val_g, y_val_g, edge_index, edge_weight, epochs=20)


Epoch 1/20 - train MAE: 0.4624 - val MAE (sample): 0.4483
Epoch 2/20 - train MAE: 0.4572 - val MAE (sample): 0.4440
Epoch 3/20 - train MAE: 0.4564 - val MAE (sample): 0.4674
Epoch 4/20 - train MAE: 0.4561 - val MAE (sample): 0.4523
Epoch 5/20 - train MAE: 0.4563 - val MAE (sample): 0.4468
Epoch 6/20 - train MAE: 0.4560 - val MAE (sample): 0.4352
Epoch 7/20 - train MAE: 0.4554 - val MAE (sample): 0.4382
Epoch 8/20 - train MAE: 0.4548 - val MAE (sample): 0.4431
Epoch 9/20 - train MAE: 0.4546 - val MAE (sample): 0.4375
Epoch 10/20 - train MAE: 0.4544 - val MAE (sample): 0.4423
Epoch 11/20 - train MAE: 0.4542 - val MAE (sample): 0.4351
Epoch 12/20 - train MAE: 0.4541 - val MAE (sample): 0.4340
Epoch 13/20 - train MAE: 0.4540 - val MAE (sample): 0.4542
Epoch 14/20 - train MAE: 0.4541 - val MAE (sample): 0.4475
Epoch 15/20 - train MAE: 0.4540 - val MAE (sample): 0.4337
Epoch 16/20 - train MAE: 0.4541 - val MAE (sample): 0.4366
Epoch 17/20 - train MAE: 0.4540 - val MAE (sample): 0.4488
Epoch 

In [ ]:
def evaluate_gcn(model, X_np, y_np, edge_index, edge_weight, mean, std, device, batch_size=64):
    model.eval()
    preds_all, actuals_all = [], []
    with torch.no_grad():
        for i in range(0, X_np.shape[0], batch_size):
            idx = np.arange(i, min(i+batch_size, X_np.shape[0]))
            batch = make_batch(X_np, y_np, idx, edge_index, edge_weight).to(device)
            out = model(batch.x, batch.edge_index, batch.edge_attr)
            preds_all.append(out.cpu().numpy())
            actuals_all.append(batch.y.cpu().numpy())
    preds = np.concatenate(preds_all) * std + mean
    actuals = np.concatenate(actuals_all) * std + mean
    mae = np.mean(np.abs(preds - actuals))
    rmse = np.sqrt(np.mean((preds - actuals)**2))
    return mae, rmse

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mae_gcn, rmse_gcn = evaluate_gcn(model, X_test_g, y_test_g, edge_index, edge_weight, mean, std, device)
print(f"GCN (100% data) -> MAE: {mae_gcn:.3f}, RMSE: {rmse_gcn:.3f}")


GCN (100% data) -> MAE: 6.115, RMSE: 9.886


## Section 11: Residual / Change (`TrafficGCN-Delta`) Formulation for Temporal Momentum
In real-world traffic forecasting, speed exhibits strong temporal inertia: current speed $x_{t-1}$ is already an effective first-order predictor of $y_t$.
Rather than learning absolute speeds from scratch, we formulate a **residual / change prediction task**:
$$\Delta y_t = y_t - x_{t-1}$$
The network models the velocity delta $\Delta y$, and the final prediction is reconstructed as:
$$\hat{y}_t = x_{t-1} + \Delta \hat{y}_t$$
This formulation stabilizes gradient flow and prevents the model from settling into an over-smoothed mean prediction.

### 11.1 Delta Batch Constructor & Residual Training Procedure


In [ ]:
def make_batch_delta(X_np, y_np, indices, edge_index, edge_weight):
    data_list = []
    for i in indices:
        x = torch.tensor(X_np[i], dtype=torch.float)      # (207, 12)
        last_val = x[:, -1]                                 # (207,) most recent speed per node
        y_delta = torch.tensor(y_np[i], dtype=torch.float) - last_val  # predict the CHANGE
        data_list.append(Data(x=x, edge_index=edge_index, edge_attr=edge_weight,
                               y=y_delta, last_val=last_val))
    return Batch.from_data_list(data_list)

def train_gcn_delta(X_train_g, y_train_g, X_val_g, y_val_g, edge_index, edge_weight,
                     epochs=30, batch_size=64, lr=0.001, hidden=64, device=None):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TrafficGCN(in_channels=X_train_g.shape[2], hidden_channels=hidden, out_channels=1).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    edge_index_d = edge_index.to(device)
    edge_weight_d = edge_weight.to(device)
    n_train = X_train_g.shape[0]

    for epoch in range(epochs):
        model.train()
        perm = np.random.permutation(n_train)
        total_loss = 0.0

        for i in range(0, n_train, batch_size):
            idx = perm[i:i+batch_size]
            batch = make_batch_delta(X_train_g, y_train_g, idx, edge_index, edge_weight).to(device)

            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.edge_attr)  # predicted delta
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)

        avg_train_loss = total_loss / n_train

        model.eval()
        with torch.no_grad():
            val_idx = np.random.choice(X_val_g.shape[0], size=min(500, X_val_g.shape[0]), replace=False)
            val_batch = make_batch_delta(X_val_g, y_val_g, val_idx, edge_index, edge_weight).to(device)
            val_out = model(val_batch.x, val_batch.edge_index, val_batch.edge_attr)
            val_loss = criterion(val_out, val_batch.y).item()

        print(f"Epoch {epoch+1}/{epochs} - train MAE(delta): {avg_train_loss:.4f} - val MAE(delta): {val_loss:.4f}")

    return model

model = train_gcn_delta(X_train_g, y_train_g, X_val_g, y_val_g, edge_index, edge_weight, epochs=30, lr=0.001)


Epoch 1/30 - train MAE(delta): 0.2000 - val MAE(delta): 0.1884
Epoch 2/30 - train MAE(delta): 0.1924 - val MAE(delta): 0.1927
Epoch 3/30 - train MAE(delta): 0.1920 - val MAE(delta): 0.1887
Epoch 4/30 - train MAE(delta): 0.1919 - val MAE(delta): 0.1959
Epoch 5/30 - train MAE(delta): 0.1918 - val MAE(delta): 0.1944
Epoch 6/30 - train MAE(delta): 0.1918 - val MAE(delta): 0.1942
Epoch 7/30 - train MAE(delta): 0.1918 - val MAE(delta): 0.1904
Epoch 8/30 - train MAE(delta): 0.1917 - val MAE(delta): 0.1908
Epoch 9/30 - train MAE(delta): 0.1916 - val MAE(delta): 0.1876
Epoch 10/30 - train MAE(delta): 0.1917 - val MAE(delta): 0.1891
Epoch 11/30 - train MAE(delta): 0.1915 - val MAE(delta): 0.1874
Epoch 12/30 - train MAE(delta): 0.1915 - val MAE(delta): 0.1864
Epoch 13/30 - train MAE(delta): 0.1916 - val MAE(delta): 0.1891
Epoch 14/30 - train MAE(delta): 0.1915 - val MAE(delta): 0.1888
Epoch 15/30 - train MAE(delta): 0.1916 - val MAE(delta): 0.1923
Epoch 16/30 - train MAE(delta): 0.1916 - val MAE(

### 11.2 Reconstructed Evaluation Function (`evaluate_gcn_delta`)
Reconstructs predictions in the original normalized speed scale and computes de-normalized MAE and RMSE metrics against actual traffic speeds.


In [ ]:
def evaluate_gcn_delta(model, X_np, y_np, edge_index, edge_weight, mean, std, device, batch_size=64):
    model.eval()
    preds_all, actuals_all = [], []
    with torch.no_grad():
        for i in range(0, X_np.shape[0], batch_size):
            idx = np.arange(i, min(i+batch_size, X_np.shape[0]))
            batch = make_batch_delta(X_np, y_np, idx, edge_index, edge_weight).to(device)
            delta_pred = model(batch.x, batch.edge_index, batch.edge_attr)
            pred = batch.last_val + delta_pred          # reconstruct actual predicted speed (normalized)
            actual = batch.last_val + batch.y            # reconstruct actual true speed (normalized)
            preds_all.append(pred.cpu().numpy())
            actuals_all.append(actual.cpu().numpy())
    preds = np.concatenate(preds_all) * std + mean
    actuals = np.concatenate(actuals_all) * std + mean
    mae = np.mean(np.abs(preds - actuals))
    rmse = np.sqrt(np.mean((preds - actuals)**2))
    return mae, rmse, preds
